# Task 4 — Sales Prediction using Python
**CodeAlpha Data Science Internship**

In this task, we predict future **Sales** based on advertising spend across 3 channels:
- **TV** — amount spent on TV advertising
- **Radio** — amount spent on Radio advertising
- **Newspaper** — amount spent on Newspaper advertising

We will use **Linear Regression** to understand how advertising spend affects sales and predict future sales.

## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All libraries imported successfully!')

## Step 2 — Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Advertising.csv', index_col=0)

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic statistics
print('Basic Statistics:')
df.describe()

In [ ]:
# Check for missing values
print('Missing values in each column:')
print(df.isnull().sum())
print('\nNo missing values — data is clean and ready!')

## Step 3 — Visualise the Data

In [ ]:
# Distribution of all columns
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cols = ['TV', 'Radio', 'Newspaper', 'Sales']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for ax, col, color in zip(axes, cols, colors):
    ax.hist(df[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.suptitle('Distribution of Advertising Spend and Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots — each channel vs Sales
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
channels = ['TV', 'Radio', 'Newspaper']
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, channel, color in zip(axes, channels, colors):
    ax.scatter(df[channel], df['Sales'], color=color, alpha=0.6, edgecolors='white', linewidth=0.5)
    # Add trend line
    z = np.polyfit(df[channel], df['Sales'], 1)
    p = np.poly1d(z)
    ax.plot(sorted(df[channel]), p(sorted(df[channel])), color='red', linewidth=2, linestyle='--')
    ax.set_title(f'{channel} vs Sales', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'{channel} Advertising Spend')
    ax.set_ylabel('Sales')

plt.suptitle('Advertising Spend vs Sales for Each Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Correlation Heatmap — Advertising vs Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with Sales:')
print(df.corr()['Sales'].sort_values(ascending=False))

In [ ]:
# Pairplot
sns.set_style('whitegrid')
sns.pairplot(df, diag_kind='kde', plot_kws={'alpha': 0.6, 'color': '#2196F3'})
plt.suptitle('Pairplot of All Features', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 4 — Prepare Data for Training

In [ ]:
# Features (X) and target (y)
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

## Step 5 — Train the Models

In [ ]:
# --- Model 1: Linear Regression ---
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

lr_r2  = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

# --- Model 2: Random Forest Regressor ---
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_r2  = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print('Model Performance:')
print('='*50)
print(f'Linear Regression  → R²: {lr_r2:.2f} | MAE: {lr_mae:.2f} | RMSE: {lr_rmse:.2f}')
print(f'Random Forest      → R²: {rf_r2:.2f} | MAE: {rf_mae:.2f} | RMSE: {rf_rmse:.2f}')

## Step 6 — Evaluate the Models

In [ ]:
# Actual vs Predicted — both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, title, r2, color in zip(
    axes,
    [lr_pred, rf_pred],
    ['Linear Regression', 'Random Forest'],
    [lr_r2, rf_r2],
    ['#2196F3', '#4CAF50']
):
    ax.scatter(y_test, pred, color=color, alpha=0.7, edgecolors='white')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', linewidth=2, label='Perfect Prediction')
    ax.set_title(f'{title}\nR² Score: {r2:.2f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Sales')
    ax.set_ylabel('Predicted Sales')
    ax.legend()

plt.suptitle('Actual vs Predicted Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Which advertising channel impacts sales the most?
coefficients = pd.DataFrame({
    'Channel': ['TV', 'Radio', 'Newspaper'],
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(coefficients['Channel'], coefficients['Coefficient'],
               color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='white', width=0.4)
for bar, val in zip(bars, coefficients['Coefficient']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

plt.title('Impact of Each Advertising Channel on Sales\n(Linear Regression Coefficients)',
          fontsize=12, fontweight='bold')
plt.ylabel('Coefficient Value')
plt.tight_layout()
plt.show()

print('\nCoefficients (impact on Sales):')
print(coefficients.to_string(index=False))
print(f'\nIntercept: {lr_model.intercept_:.2f}')

In [ ]:
# Model comparison chart
metrics = ['R² Score', 'MAE', 'RMSE']
lr_scores = [lr_r2, lr_mae, lr_rmse]
rf_scores = [rf_r2, rf_mae, rf_rmse]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, lr_scores, width, label='Linear Regression', color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, rf_scores, width, label='Random Forest', color='#4CAF50', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Model Comparison — Linear Regression vs Random Forest', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
plt.tight_layout()
plt.show()

## Step 7 — Business Insights & Conclusion

### What we found:

1. **TV advertising has the highest impact on sales** — TV has the strongest positive correlation with sales (0.78). Every unit increase in TV spend leads to the highest increase in sales compared to other channels.

2. **Radio is the second most effective channel** — Radio advertising also has a strong positive impact on sales with a correlation of 0.58.

3. **Newspaper has the least impact** — Newspaper advertising has the weakest correlation with sales (0.23), meaning spending more on newspaper ads does not significantly boost sales.

4. **High R² Score** — The Linear Regression model achieved a strong R² score meaning it can explain most of the variation in sales based on advertising spend.

5. **Random Forest performed better** — Random Forest gave a higher R² and lower error than Linear Regression, showing that the relationship between advertising and sales is slightly non-linear.

### Business Recommendation:
> **Invest more in TV and Radio advertising and reduce Newspaper spend** — this will give the highest return on investment for sales growth.

### Conclusion:
Machine learning can accurately predict sales based on advertising spend. This helps businesses make smarter decisions about where to invest their marketing budget for maximum sales impact.